In [22]:
import pandas as pd
from tqdm import tqdm


train_df = pd.read_csv('data/processed_data/train.csv')
test_df = pd.read_csv('data/processed_data/test.csv')
ports_df = pd.read_csv('data/original_data/ports.csv', sep='|')

In [23]:
port_counts = train_df['portId'].value_counts()

# Identify portIds with fewer than 2 samples
rare_ports = port_counts[port_counts < 2].index.tolist()

print(f"Rare portIds with fewer than 2 samples: {rare_ports}")

Rare portIds with fewer than 2 samples: ['634c4de270937fc01c3a713e', '634c4de270937fc01c3a7342', '61d3723a3aeaecc07011a441', '635e9424bef885201a5e0fcc', '634c4de270937fc01c3a7425', '634c4de270937fc01c3a78e9', '634c4de270937fc01c3a74f8', '634c4de270937fc01c3a74c6', '634c4de270937fc01c3a75be', '634c4de270937fc01c3a75d9', '635edab5bef885201a5e0fd4', '61d3773893c6feb83e5eb5bf', '634c4de270937fc01c3a7005', '634c4de270937fc01c3a71a7', '634c4de270937fc01c3a7379', '61d37a131366c3998241d90b', '634c4de270937fc01c3a7506', '634c4de270937fc01c3a75a3', '634c4de270937fc01c3a7345', '61d37a301366c3998241d943']


In [24]:
train_df = train_df[~train_df['portId'].isin(rare_ports)].copy()

port_counts_cleaned = train_df['portId'].value_counts()
rare_ports_after = port_counts_cleaned[port_counts_cleaned < 2].index.tolist()

if len(rare_ports_after) == 0:
    print("All portId classes have at least 2 samples.")
else:
    print(f"Still rare portIds with fewer than 2 samples: {rare_ports_after}")


All portId classes have at least 2 samples.


In [25]:

num_null_train = train_df['portId'].isnull().sum()
print(f"Number of null values in 'portId' in train_df: {num_null_train}")

Number of null values in 'portId' in train_df: 1615


In [26]:
train_df = train_df.sort_values(by=['vesselId', 'time'])

train_df['portId'] = train_df.groupby('vesselId')['portId'].ffill()
train_df['portId'] = train_df.groupby('vesselId')['portId'].bfill()

num_null_after = train_df['portId'].isnull().sum()
print(f"Number of null values in 'portId' after filling: {num_null_after}")

Number of null values in 'portId' after filling: 0


In [27]:
train_df.columns

Index(['time', 'cog', 'sog', 'rot', 'heading', 'navstat', 'etaRaw', 'latitude',
       'longitude', 'vesselId', 'portId', 'latitude_1_steps_ago',
       'longitude_1_steps_ago', 'time_position_1_steps_ago',
       'latitude_2_steps_ago', 'longitude_2_steps_ago',
       'time_position_2_steps_ago', 'latitude_3_steps_ago',
       'longitude_3_steps_ago', 'time_position_3_steps_ago',
       'latitude_4_steps_ago', 'longitude_4_steps_ago',
       'time_position_4_steps_ago', 'latitude_5_steps_ago',
       'longitude_5_steps_ago', 'time_position_5_steps_ago',
       'max_lat_change_last_5_steps', 'min_lat_change_last_5_steps',
       'max_long_change_last_5_steps', 'min_long_change_last_5_steps',
       'cog_1_step_ago', 'time_cog_1_step_ago', 'cog_2_steps_ago',
       'time_cog_2_steps_ago', 'hours_passed', 'hour_sin', 'hour_cos',
       'minute_sin', 'minute_cos', 'week_of_the_year', 'day_of_the_year',
       'time_diff_gt_10min', 'time_diff_gt_20min', 'time_diff_gt_40min',
       'time_d

In [28]:
features = ['time', 'cog', 'latitude_1_steps_ago',
       'longitude_1_steps_ago', 'time_position_1_steps_ago',
       'latitude_2_steps_ago', 'longitude_2_steps_ago',
       'time_position_2_steps_ago', 'latitude_3_steps_ago',
       'longitude_3_steps_ago', 'time_position_3_steps_ago',
       'latitude_4_steps_ago', 'longitude_4_steps_ago',
       'time_position_4_steps_ago', 'latitude_5_steps_ago',
       'longitude_5_steps_ago', 'time_position_5_steps_ago',
       'max_lat_change_last_5_steps', 'min_lat_change_last_5_steps',
       'max_long_change_last_5_steps', 'min_long_change_last_5_steps',
       'cog_1_step_ago', 'time_cog_1_step_ago', 'cog_2_steps_ago',
       'time_cog_2_steps_ago', 'hours_passed', 'hour_sin', 'hour_cos',
       'minute_sin', 'minute_cos', 'week_of_the_year', 'day_of_the_year',
       'time_diff_gt_10min', 'time_diff_gt_20min', 'time_diff_gt_40min',
       'time_diff_gt_1hour', 'time_diff_gt_2hours', 'time_diff_gt_6hours',
       'time_diff_gt_12hours', 'time_diff_gt_1day', 'lat_change_2_to_1_steps',
       'lat_change_3_to_2_steps', 'lat_change_4_to_3_steps',
       'lat_change_5_to_4_steps', 'lon_change_2_to_1_steps',
       'lon_change_3_to_2_steps', 'lon_change_4_to_3_steps',
       'lon_change_5_to_4_steps', 'avg_lat_change_1_steps',
       'avg_lat_change_2_steps', 'avg_lat_change_3_steps',
       'avg_lat_change_4_steps', 'avg_lat_change_5_steps',
       'avg_lon_change_1_steps', 'avg_lon_change_2_steps',
       'avg_lon_change_3_steps', 'avg_lon_change_4_steps',
       'avg_lon_change_5_steps', 'vesselType_14.0', 'vesselType_21.0',
       'vesselType_83.0', 'enginePower', 'CEU', 'GT', 'breadth', 'length',
       'DWT', 'maxSpeed']

In [29]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns

In [30]:
X = pd.DataFrame(train_df, columns=features)
y = train_df['portId']

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(train_df["portId"])

y = y_encoded

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [31]:
num_classes = len(label_encoder.classes_)

print(num_classes)

752


In [ ]:

xgb_multiclass_params = {
    'objective': 'multi:softprob',  # For multiclass classification
    'num_class': num_classes,       # Number of classes
    'learning_rate': 0.1,
    'max_depth': 6,
    'n_estimators': 100,            # Number of boosting rounds
    'eval_metric': 'mlogloss',      # Multi-class log loss
    'use_label_encoder': False,     # To suppress a warning
    'random_state': 42
}

xgb_clf = xgb.XGBClassifier(**xgb_multiclass_params)

eval_set = [(X_train, y_train), (X_test, y_test)]

xgb_clf.fit(
    X_train, y_train,
    early_stopping_rounds=10,
    eval_set=eval_set,
    verbose=True
)

/opt/homebrew/lib/python3.11/site-packages/xgboost/sklearn.py:889: UserWarning: `early_stopping_rounds` in `fit` method is deprecated for better compatibility with scikit-learn, use `early_stopping_rounds` in constructor or`set_params` instead.
  warnings.warn(


[0]	validation_0-mlogloss:3.02557	validation_1-mlogloss:3.03253
[1]	validation_0-mlogloss:9.09660	validation_1-mlogloss:9.13571
[2]	validation_0-mlogloss:13.90252	validation_1-mlogloss:13.91573


In [ ]:
y_pred = xgb_clf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")